In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from NOTEBOOKS.calculateEVS import *
from MODELS.pipeline import *
from MODELS.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Load Model

In [18]:
# Load split NGBoost models (mean, variance, and calibration factor)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTION.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTION.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor)  
features = joblib.load('../MODELS/SAVED_MODELS/feature_list.pkl')

print(f"Loaded models with calibration factor: {calibration_factor}")

Loaded models with calibration factor: 1.4


### Load Player Data and Bookmaker Data

In [25]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s25= pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv').sort_values(by='GAME_DATE')
s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')
df = pd.concat([s25, s26])

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

### Update projected starting lineups

In [20]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated c:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor\MODELS/teamInfo.py
Updated 22 teams with confirmed lineups


### Top EVs for single bets

In [9]:
singlePTSBookies = usData[(usData['CATEGORY'] == 'player_points') & (usData['ODDS'] <= 250) & (usData['ODDS'] >= -250)]

results = calculateSingleBets(df, singlePTSBookies, model, features, current_date, 
                             edge_threshold=0.20, stake=100, 
                             variance_inflation=1.1, distribution_type='t', 
                             use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)  

singleBets = results
singleBets = singleBets[(singleBets['SIGMA FLAG'] == 'Med') | (singleBets['SIGMA FLAG'] == 'Low')].sort_values(by='EV$', ascending=False).reset_index(drop=True)
singleBets = singleBets[['NAME', 'BOOKMAKER','LINE', 'PREDICTION','SIDE','ODDS','RECOMMENDATION', 'EV$', 'EXPECTED ROI', 'KELLY_FRACTION','SIGMA FLAG']]
singleBets.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/singleBets.csv', index=False)
singleBets.sample(5)

Processing single bets with single model...


,NAME,BOOKMAKER,LINE,PREDICTION,SIDE,ODDS,RECOMMENDATION,EV$,EXPECTED ROI,KELLY_FRACTION,SIGMA FLAG
299,Kevon Looney,DraftKings,4.5,8.14,Under,-114,0,-62.53,-62.5,0.000,Med
357,Neemias Queta,Bovada,6.5,14.05,Under,190,0,-84.66,-84.7,0.000,Low
138,Luguentz Dort,Bovada,9.5,10.64,Over,115,0,25.52,25.5,0.222,Med
186,Walter Clayton Jr.,BetOnline.ag,6.5,6.50,Under,-111,0,-3.59,-3.6,0.000,Low
33,Draymond Green,DraftKings,9.5,16.37,Over,-117,1,68.74,68.7,0.804,Med


## Top EVs for 2 leg bets

### Underdog picks

In [26]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogPairs = results
underdogPairs = underdogPairs[
    underdogPairs[['SIGMA FLAG 1', 'SIGMA FLAG 2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Jock Landale,Isaiah Hartenstein,8.5,10.5,over,over,1,13.54,0.677,Med,Med
1,Neemias Queta,Jock Landale,8.5,8.5,over,over,1,12.72,0.636,Low,Med
2,Neemias Queta,Isaiah Hartenstein,8.5,10.5,over,over,1,12.48,0.624,Low,Med
3,Caris LeVert,Jock Landale,10.5,8.5,under,over,1,11.04,0.552,Low,Med
4,Mitchell Robinson,Jock Landale,4.5,8.5,over,over,1,10.81,0.541,Low,Med


### Prizepicks picks

In [27]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

results = calculate2LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.35, stake=10, 
                     variance_inflation=1.1, distribution_type='t',
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)
pairsPrizepicks = results
pairsPrizepicks = pairsPrizepicks[
    pairsPrizepicks[['SIGMA FLAG 1', 'SIGMA FLAG 2']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV$', ascending=False).reset_index(drop=True)
pairsPrizepicks = pairsPrizepicks[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
pairsPrizepicks.head()

,NAME 1,NAME 2,LINE 1,LINE 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV$,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
0,Jock Landale,Isaiah Hartenstein,8.5,10.5,over,over,1,13.54,0.677,Med,Med
1,Neemias Queta,Jock Landale,8.5,8.5,over,over,1,12.72,0.636,Low,Med
2,Neemias Queta,Isaiah Hartenstein,8.5,10.5,over,over,1,12.48,0.624,Low,Med
3,Jock Landale,Jose Alvarado,8.5,9.5,over,under,1,11.34,0.567,Med,Low
4,Jose Alvarado,Isaiah Hartenstein,9.5,10.5,under,over,1,11.11,0.556,Low,Med


## 3 leg parlay

### Underdog picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

threeLeg = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.10, stake=10, 
                     variance_inflation=1.1, distribution_type='t', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

underdogTrios = threeLeg
underdogTrios = threeLeg[
    threeLeg[['SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV$', ascending=False).reset_index(drop=True)
underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3','MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,EXPECTED ROI,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Neemias Queta,Jock Landale,Isaiah Hartenstein,8.5,8.5,10.5,14.05,16.89,18.43,over,over,over,1,279.75,279.7,0.559,Low,Med,Med
1,Caris LeVert,Jock Landale,Isaiah Hartenstein,10.5,8.5,10.5,6.81,16.89,18.43,under,over,over,1,251.66,251.7,0.503,Low,Med,Med
2,Mitchell Robinson,Jock Landale,Isaiah Hartenstein,4.5,8.5,10.5,8.25,16.89,18.43,over,over,over,1,247.89,247.9,0.496,Low,Med,Med
3,Jock Landale,Josh Okogie,Isaiah Hartenstein,8.5,8.5,10.5,16.89,12.92,18.43,over,over,over,1,246.87,246.9,0.494,Med,Med,Med
4,Jabari Walker,Jock Landale,Isaiah Hartenstein,4.5,8.5,10.5,8.90,16.89,18.43,over,over,over,1,242.59,242.6,0.485,Med,Med,Med


### Prizepicks picks

In [ ]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, edge_threshold=0.40, stake=100, 
                     variance_inflation=1.1, distribution_type='normal', 
                     use_monte_carlo=True, n_simulations=10000, max_kelly=0.25)

triosPrizepicks = threeLeg[
    threeLeg[['SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].isin(['Med', 'Low']).all(axis=1)
].sort_values(by='EV$', ascending=False).reset_index(drop=True)
triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV$', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']]
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV$,EXPECTED ROI,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
0,Neemias Queta,Jock Landale,Isaiah Hartenstein,8.5,8.5,10.5,14.05,16.89,18.43,over,over,over,1,255.50,255.5,0.511,Low,Med,Med
1,Adem Bona,Jock Landale,Isaiah Hartenstein,4.5,8.5,10.5,8.81,16.89,18.43,over,over,over,1,230.41,230.4,0.461,Low,Med,Med
2,Caris LeVert,Jock Landale,Isaiah Hartenstein,10.5,8.5,10.5,6.81,16.89,18.43,under,over,over,1,224.52,224.5,0.449,Low,Med,Med
3,Jock Landale,Josh Okogie,Isaiah Hartenstein,8.5,8.5,10.5,16.89,12.92,18.43,over,over,over,1,221.22,221.2,0.442,Med,Med,Med
4,Mitchell Robinson,Jock Landale,Isaiah Hartenstein,4.5,8.5,10.5,8.25,16.89,18.43,over,over,over,1,219.82,219.8,0.440,Low,Med,Med


In [29]:
def count_line_hits(player_df, line, category, game_windows=[5, 10, 15]):
    results = {}
    player_df_sorted = player_df.sort_values('GAME_DATE')
    total_games = len(player_df_sorted)

    for window in game_windows:
        # Handle players with fewer games
        if total_games < window:
            last_n_games = player_df_sorted
        else:
            last_n_games = player_df_sorted.tail(window)

        if category == 'player_points':
            hits = (last_n_games['PTS'] > line).sum()
        elif category == 'player_assists':
            hits = (last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds':
            hits = (last_n_games['REB'] > line).sum()
        elif category == 'player_threes':
            hits = (last_n_games['FG3M'] > line).sum()
        elif category == 'player_blocks':
            hits = (last_n_games['BLK'] > line).sum()
        elif category == 'player_steals':
            hits = (last_n_games['STL'] > line).sum()
        elif category == 'player_field_goals':
            hits = (last_n_games['FGM'] > line).sum()
        elif category == 'player_frees_made':
            hits = (last_n_games['FTM'] > line).sum()
        elif category == 'player_points_rebounds_assists':
            hits = (last_n_games['PTS'] + last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_points_rebounds':
            hits = (last_n_games['PTS'] + last_n_games['REB'] > line).sum()
        elif category == 'player_points_assists':
            hits = (last_n_games['PTS'] + last_n_games['AST'] > line).sum()
        elif category == 'player_rebounds_assists':
            hits = (last_n_games['REB'] + last_n_games['AST'] > line).sum()
        elif category == 'player_turnovers':
            hits = (last_n_games['TOV'] > line).sum()
        else:
            hits = 0

        results['NAME'] = player_df_sorted['PLAYER_NAME'].iloc[0] if total_games > 0 else 'Unknown'
        results['CATEGORY'] = category
        results['LINE'] = line
        results[f'L-{window}'] = round(hits / window, 2)


    return results

prizePicks = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks')]
prizePicks= prizePicks.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
prizePicks

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Pascal Siakam,Over,26.5,-137,2025-11-06,2025-11-05T23:40:16Z
2,PrizePicks,player_points,Cam Thomas,Over,24.5,-137,2025-11-06,2025-11-05T23:40:16Z
4,PrizePicks,player_points,Michael Porter Jr,Over,18.5,-137,2025-11-06,2025-11-05T23:40:16Z
6,PrizePicks,player_points,Aaron Nesmith,Over,17.5,-137,2025-11-06,2025-11-05T23:40:16Z
8,PrizePicks,player_points,Jarace Walker,Over,15.5,-137,2025-11-06,2025-11-05T23:40:16Z
...,...,...,...,...,...,...,...,...
3326,PrizePicks,player_blocks_steals,Deandre Ayton,Over,1.5,-137,2025-11-06,2025-11-05T23:40:01Z
3328,PrizePicks,player_blocks_steals,Luka Doncic,Over,1.5,-137,2025-11-06,2025-11-05T23:40:01Z
3330,PrizePicks,player_blocks_steals,Jarred Vanderbilt,Over,1.5,-137,2025-11-06,2025-11-05T23:40:01Z
3332,PrizePicks,player_blocks_steals,Shai Gilgeous-Alexander,Over,2.5,-137,2025-11-06,2025-11-05T23:39:52Z


In [30]:
line_hit_data = []

for index, row in prizePicks.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_PRIZEPICKS'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

        
        

Saved 136 records for player_points to player_points.csv
Saved 69 records for player_rebounds to player_rebounds.csv
Saved 35 records for player_assists to player_assists.csv
Saved 24 records for player_threes to player_threes.csv
Saved 8 records for player_blocks to player_blocks.csv
Saved 19 records for player_steals to player_steals.csv
Saved 46 records for player_field_goals to player_field_goals.csv
Saved 32 records for player_frees_made to player_frees_made.csv
Saved 9 records for player_frees_attempts to player_frees_attempts.csv
Saved 134 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 131 records for player_points_rebounds to player_points_rebounds.csv
Saved 120 records for player_points_assists to player_points_assists.csv
Saved 73 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 23 records for player_turnovers to player_turnovers.csv
Saved 23 records for player_blocks_steals to player_blocks_steals.csv

All categ

In [31]:
underdog = dfsData[(dfsData['BOOKMAKER'] == 'Underdog')]
underdog= underdog.drop_duplicates(subset=['CATEGORY', 'NAME', 'LINE'], keep='first')
line_hit_data = []

for index, row in underdog.iterrows():
    name = row['NAME']
    line = row['LINE']
    category = row['CATEGORY']
    player_df = df[df['PLAYER_NAME'] == name]
    
    if len(player_df) > 0:
        hit_counts = count_line_hits(player_df, line, category)
        line_hit_data.append(hit_counts)

line_hit_df = pd.DataFrame(line_hit_data)



over_rates_dir = '../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG'
os.makedirs(over_rates_dir, exist_ok=True)
today = datetime.today().strftime('%Y%m%d')

for category in line_hit_df['CATEGORY'].unique():
    # Filter data for this category
    category_data = line_hit_df[line_hit_df['CATEGORY'] == category]
    
    filename = f"{category}.csv"
    filepath = os.path.join(over_rates_dir, filename)
    category_data.to_csv(filepath, index=False)
    print(f"Saved {len(category_data)} records for {category} to {filename}")

print(f"\nAll category files saved to {over_rates_dir}")

Saved 113 records for player_points to player_points.csv
Saved 39 records for player_rebounds to player_rebounds.csv
Saved 16 records for player_assists to player_assists.csv
Saved 21 records for player_threes to player_threes.csv
Saved 1 records for player_blocks to player_blocks.csv
Saved 7 records for player_steals to player_steals.csv
Saved 2 records for player_frees_made to player_frees_made.csv
Saved 131 records for player_points_rebounds_assists to player_points_rebounds_assists.csv
Saved 58 records for player_points_rebounds to player_points_rebounds.csv
Saved 49 records for player_points_assists to player_points_assists.csv
Saved 30 records for player_rebounds_assists to player_rebounds_assists.csv
Saved 5 records for player_turnovers to player_turnovers.csv
Saved 4 records for player_blocks_steals to player_blocks_steals.csv

All category files saved to ../DATA/CSV_FILES/PROP_DATA/OVER_RATES_UNDERDOG
